# Load Libraries

In [ ]:
!pip install -qqq ragas langchain_huggingface PyPDF2 faiss-cpu evaluate langchain-text-splitters rank-bm25 chromadb wtpsplit rapidfuzz sentence_transformers datasets langchain_openai

In [ ]:
from datasets import load_dataset
from huggingface_hub import login, snapshot_download, InferenceClient
import pandas as pd
import numpy as np
import os
from pathlib import Path
import difflib
import asyncio
from rich.progress import track

import PyPDF2
import requests
from sentence_transformers import SentenceTransformer, CrossEncoder
import faiss
import re
from evaluate import load
from rank_bm25 import BM25Okapi
import chromadb
import unicodedata
from torch.utils.data import DataLoader
from wtpsplit import SaT
from rapidfuzz import fuzz
import ast
from transformers import AutoTokenizer, pipeline

from ragas import evaluate
from ragas.metrics import Faithfulness, AnswerCorrectness
from datasets import Dataset
from dataclasses import dataclass
from typing import Optional
from langchain_community.embeddings import HuggingFaceEmbeddings

/usr/local/lib/python3.12/dist-packages/google/colab/html/_background_server.py:103: DeprecationWarning: make_current is deprecated; start the event loop first
  ioloop.make_current()
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/tmp/ipykernel_1355/2472036751.py:26: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, AnswerCorrectness
/tmp/ipykernel_1355/2472036751.py:26: DeprecationWarning: Importing AnswerCorrectness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collec

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
%cd /content/drive/MyDrive/RAG
cwd = os.getcwd()

/content/drive/MyDrive/RAG


# Load Dataset

In [ ]:
hf_token = '**'
login(hf_token)

In [ ]:
# snapshot_download(
#         repo_id='theatticusproject/cuad',
#         repo_type="dataset",
#         local_dir=f'{cwd}/data',
#         ignore_patterns=["*.gitattributes", ".gitignore"],
#         token = hf_token
#     )

Fetching 744 files:   0%|          | 0/744 [00:00<?, ?it/s]

'/content/drive/MyDrive/RAG/data'

In [ ]:
def pdf_files_paths_list(pdf_contracts_dir):
    pdf_files_paths = []
    for folder in Path(pdf_contracts_dir).iterdir():
        base = folder / os.listdir(folder)[0] if folder.name == 'Part_II' else folder
        for sub_folder in base.iterdir():
            pdf_files_paths.extend(sub_folder / pdf for pdf in os.listdir(sub_folder))
    return pdf_files_paths

In [ ]:
def txt_files_paths_list(txt_contracts_dir):
    txt_files_paths = []
    for folder in Path(txt_contracts_dir).iterdir():
        for txt_file in folder.iterdir():
            txt_files_paths.append(txt_file)
    return txt_files_paths

In [ ]:
data_folder = f'{cwd}/data/CUAD_v1'
pdf_contracts = f'{data_folder}/full_contract_pdf'
txt_contracts = f'{data_folder}/full_contract_txt'

In [ ]:
pdf_files_paths = pdf_files_paths_list(pdf_contracts)
assert len(pdf_files_paths) == 510

In [ ]:
txt_files_paths = txt_files_paths_list(txt_contracts)
len(txt_files_paths)

200

In [ ]:
master_clauses = pd.read_csv(f'{data_folder}/master_clauses.csv')
master_clauses.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Filename,Document Name,Document Name-Answer,Parties,Parties-Answer,Agreement Date,Agreement Date-Answer,Effective Date,Effective Date-Answer,Expiration Date,...,Liquidated Damages,Liquidated Damages-Answer,Warranty Duration,Warranty Duration-Answer,Insurance,Insurance-Answer,Covenant Not To Sue,Covenant Not To Sue-Answer,Third Party Beneficiary,Third Party Beneficiary-Answer
0,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,['MARKETING AFFILIATE AGREEMENT'],MARKETING AFFILIATE AGREEMENT,"['BIRCH FIRST GLOBAL INVESTMENTS INC.', 'MA', ...","Birch First Global Investments Inc. (""Company""...","['8th day of May 2014', 'May 8, 2014']",5/8/2014,['This agreement shall begin upon the date of ...,NaN,['This agreement shall begin upon the date of ...,...,[],No,"[""COMPANY'S SOLE AND EXCLUSIVE LIABILITY FOR T...",Yes,[],No,[],No,[],No
1,EuromediaHoldingsCorp_20070215_10SB12G_EX-10.B...,['VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT'],VIDEO-ON-DEMAND CONTENT LICENSE AGREEMENT,"['EuroMedia Holdings Corp.', 'Rogers', 'Rogers...","Rogers Cable Communications Inc. (""Rogers""); E...","['July 11 , 2006']",7/11/2006,"['July 11 , 2006']",7/11/2006,"['The term of this Agreement (the ""Initial Ter...",...,[],No,[],No,[],No,[],No,[],No
2,FulucaiProductionsLtd_20131223_10-Q_EX-10.9_83...,['CONTENT DISTRIBUTION AND LICENSE AGREEMENT'],CONTENT DISTRIBUTION AND LICENSE AGREEMENT,"['Producer', 'Fulucai Productions Ltd.', 'Conv...","CONVERGTV, INC. (“ConvergTV”); Fulucai Product...","['November 15, 2012']",11/15/2012,"['November 15, 2012']",11/15/2012,[],...,[],No,[],No,[],No,[],No,[],No
3,GopageCorp_20140221_10-K_EX-10.1_8432966_EX-10...,['WEBSITE CONTENT LICENSE AGREEMENT'],WEBSITE CONTENT LICENSE AGREEMENT,"['PSiTech Corporation', 'Licensor', 'Licensee'...","PSiTech Corporation (""Licensor""); Empirical Ve...","['Feb 10, 2014']",2/10/2014,"['Feb 10, 2014']",2/10/2014,['The initial term of this Agreement commences...,...,[],No,[],No,[],No,[],No,[],No
4,IdeanomicsInc_20160330_10-K_EX-10.26_9512211_E...,['CONTENT LICENSE AGREEMENT'],CONTENT LICENSE AGREEMENT,"['YOU ON DEMAND HOLDINGS, INC.', 'Licensor', '...",Beijing Sun Seven Stars Culture Development Li...,"['December 21, 2015']",12/21/2015,"['December 21, 2015']",12/21/2015,"['The Term of this Agreement (the ""Term"") shal...",...,[],No,[],No,[],No,[],No,[],No


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
master_clauses.shape

(510, 83)

In [ ]:
label_groups = ['Filename']
folder = 'label_group_xlsx'
for f in os.listdir(f'{data_folder}/{folder}'):
    if f == 'Label Report - Uncapped Liability (Group 5).xlsx':
        label_groups.append('Uncapped Liability')
    else:
        df = pd.read_excel(f'{data_folder}/{folder}/{f}')
        labels = list(df.columns.values[1:])
        for label in labels:
            if 'Answer' in label:
              label_groups.pop()
            label_groups.append(label)
        # print(label, f)

In [ ]:
assert len(label_groups) == 41

In [ ]:
clauses = [clause.lower() for clause in master_clauses.columns.values]
for label in label_groups:
    if label.lower() not in clauses:
        match_str = difflib.get_close_matches(label.lower(), clauses, n=1)
        print(match_str, label.lower())

['rofr/rofo/rofn'] rofr-rofo-rofn
['revenue/profit sharing'] revenue-profit sharing
['unlimited/all-you-can-eat-license'] unlimited/all-you-can-eat license


# RAG Pipeline

#### Loading Language Models

In [ ]:
%%capture
text_splitter = SaT("sat-3l")
embedding_model_name = 'BAAI/bge-large-en-v1.5'
embedding_model = SentenceTransformer(embedding_model_name)
cross_encoder_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-12-v2')
tokenizer = AutoTokenizer.from_pretrained(embedding_model_name)

chat_model = "Qwen/Qwen2.5-1.5B-Instruct"
# pipe = pipeline(
#     "text-generation",
#     model=chat_model,
#     device_map="auto",
#     max_new_tokens=50,
#     do_sample=False,
# )

# free_embeddings = HuggingFaceEmbeddings(model_name=embedding_model_name)

SubwordXLMForTokenClassification LOAD REPORT from: segment-any-text/sat-3l
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertModel LOAD REPORT from: BAAI/bge-large-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-12-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.

In [ ]:
def extract_text_from_pdf(file_path):
    with open(file_path, 'rb') as file:
      reader = PyPDF2.PdfReader(file)
      text = ""
      for page in reader.pages:
          page_text = page.extract_text()
          if page_text:
              text += page_text + "\n"
    text = unicodedata.normalize("NFKC", text)
    return text

In [ ]:
def extract_chunks(paragraphs):
    chunks = []
    for paragraph in paragraphs:
        chunk = ''
        for sentence in paragraph:
            chunk += sentence + ' '
        chunks.append(chunk)

    return chunks

### Creating vector embeddings for single document

In [ ]:
# file_path = f'{cwd}/CreditcardscomInc_20070810_S-1_EX-10.33_362297_EX-10.33_Affiliate Agreement.pdf'
file_path = f'{cwd}/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'
# file_path = '/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Affiliate_Agreements/UsioInc_20040428_SB-2_EX-10.11_1723988_EX-10.11_Affiliate Agreement 2.pdf'
# file_path = '/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Co_Branding/MphaseTechnologiesInc_20030911_10-K_EX-10.15_1560667_EX-10.15_Co-Branding Agreement.pdf'

In [ ]:
text = extract_text_from_pdf(file_path)

In [ ]:
paragraphs = text_splitter.split(text, do_paragraph_segmentation=True)

In [ ]:
chunks = extract_chunks(paragraphs)

### Creating Vector Database

In [ ]:
os.environ["HF_TOKEN"] = hf_token
embed_client = InferenceClient()

In [ ]:
# Initialize the client (persists data to a local directory)
client = chromadb.PersistentClient(path="./cuad_vector_db_copy")
collection = client.get_collection(name = 'atticus_vector_data_cp')

In [ ]:
file_path

PosixPath('/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Transportation/RangeResourcesLouisianaInc_20150417_8-K_EX-10.5_9045501_EX-10.5_Transportation Agreement.pdf')

In [ ]:
n_start = 189
n_end = n_start + 1

for file_path in track(pdf_files_paths[n_start: n_end], 'Processing...'):
    # Create a collection (similar to a table in SQL)
    name = Path(file_path).stem.replace(" ", "-") # Replace spaces with hyphens for valid collection name

    text = extract_text_from_pdf(file_path)
    paragraphs = text_splitter.split(text, do_paragraph_segmentation=True)

    chunks = extract_chunks(paragraphs)
    # embeddings = embed_client.feature_extraction(
    # text=chunks,
    # model=embedding_model_name)
    embeddings = embedding_model.encode(chunks, convert_to_numpy=True)

    ids = [name + 'chunk_' + str(id) for id in range(len(chunks))]
    metadatas = [{'source': name}]*len(chunks)

    collection.add(
        ids = ids,
        documents = chunks,
        embeddings = embeddings,
        metadatas = metadatas
    )

Output()

## Hybrid Retrieval

In [ ]:
def hybrid_search(query, query_embedding, chunks, collection, name, use_cross_enc = True, top_k=5):
    # 1. Dense retrieval
    dense_results = collection.query(query_embeddings=[query_embedding], n_results=10,
                                     where = {'source': name})
    dense_ids = dense_results["ids"][0]  # ["chunk_0", "chunk_3", ...]

    # 2. Sparse retrieval (BM25)
    tokenized_chunks = [tokenizer.tokenize(chunk) for chunk in chunks]
    bm25 = BM25Okapi(tokenized_chunks)
    tokenized_query = tokenizer.tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    top_sparse_indices = sorted(range(len(bm25_scores)), key=lambda i: bm25_scores[i], reverse=True)[:20]
    sparse_ids = [f"{name}chunk_{i}" for i in top_sparse_indices]  # convert to chunk IDs

    # 3. Merge with RRF
    def reciprocal_rank_fusion(dense_ids, sparse_ids, k=60):
        scores = {}
        for rank, doc_id in enumerate(dense_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        for rank, doc_id in enumerate(sparse_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    if use_cross_enc:
        candidate_ids = reciprocal_rank_fusion(dense_ids, sparse_ids)[:10]

        candidate_data = collection.get(ids=candidate_ids)
        candidate_texts = candidate_data["documents"]

        # # 4. CROSS-ENCODER RERANKING (Precision Stage)
        # # Pair the query with each candidate text for scoring
        rerank_pairs = [[query, text] for text in candidate_texts]
        scores = cross_encoder_model.predict(rerank_pairs)

        # 5. Pair scores with documents and sort
        doc_score_pairs = list(zip(candidate_texts, scores))
        doc_score_pairs.sort(key=lambda x: x[1], reverse=True)
        # Return only the top_k requested results
        return [text for text, score in doc_score_pairs[:top_k]]
    else:
        final_ids = reciprocal_rank_fusion(dense_ids, sparse_ids)[:top_k]
        final_texts = [collection.get(ids = id)['documents'][0] for id in final_ids]

        return  final_texts

In [ ]:
query_dict = {}
hit_rate_dict = {}
mrr_dict = {}

In [ ]:
def normalize(text: str) -> str:
    """Lowercase and strip punctuation for comparison."""
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)  # remove punctuation
    text = re.sub(r"\s+", " ", text)       # collapse spaces
    return text.strip()

In [ ]:
def is_phrase_in_text(phrase: str, text: str, threshold: int = 80) -> dict:
    norm_text   = normalize(text)
    norm_phrase = normalize(phrase)

    # partial_ratio checks if phrase is contained as a substring
    score = fuzz.partial_ratio(norm_phrase, norm_text)

    return {
        "contained": score >= threshold,
        "score": score,
        "verdict": "YES" if score >= threshold else "NO"
    }

In [ ]:
# column = 'Parties'
# column = 'Agreement Date'
column = 'Governing Law'
# column = 'Termination For Convenience'
# column = 'License Grant'
# column = 'Non-Transferable License'
# column = 'Cap On Liability'
# column = 'Effective Date'
# column = 'Expiration Date'
# column = 'Renewal Term'
# column = 'Notice Period To Terminate Renewal'
# column = 'Most Favored Nation'
# column = 'Competitive Restriction Exception'
# column = 'Non-Compete'
# column = 'Exclusivity'
# column = 'No-Solicit Of Customers'
# column = 'No-Solicit Of Employees'
# column = 'Non-Disparagement'

In [ ]:
# query = f"""{column}: Which parties/corporations/LLC/Inc. agreed upon the agreement? What are their addresses?"""
# query = f'{column}: What date was the contract dated or what is the agreement date?'
query = f'{column}: Which state/country’s law governs the interpretation of the contract?'
# query = f"{column}: Can a party terminate this contract without cause (solely by giving a notice and allowing a waiting period to expire)?"
# query = f"{column}: Does the contract contain a license granted by one party to its counterparty?"
# query = f"{column}: Does the contract limit the ability of a party to transfer the license being granted to a third party?"
# query = f"{column}: Does the contract include a cap on liability upon the breach of a party’s obligation? This includes time limitation for the counterparty to bring claims or maximum amount for recovery."
# query = f'{column}: On what date is the contract is effective?'
# query = f'{column}: On what date will the contract’s initial term expire?'
# query = f'{column}: What is the renewal term after the initial term expires? This includes automatic extensions and unilateral extensions with prior notice.'
# query = f'{column}: What is the notice period required to terminate renewal?'
# query = f"""{column}: Is there a clause that if a third party gets better terms on the licensing
# or sale of technology/goods/services described in the contract, the
# buyer of such technology/goods/services under the contract shall
# be entitled to those better terms?
# """
# query = f"""{column}: This category includes the exceptions or carveouts to Non-Compete,
# Exclusivity and No-Solicit of Customers above."""
# query = f"""{column}: Is there a restriction on the ability of a party to compete with
# the counterparty or operate in a certain geography or business or
# technology sector?"""
# query = f"""{column}: Is there an exclusive dealing commitment with the counterparty?
# This includes a commitment to procure all “requirements” from one
# party of certain technology, goods, or services or a prohibition on
# licensing or selling technology, goods or services to third parties,
# or a prohibition on collaborating or working with other parties),
# whether during the contract or after the contract ends (or both)."""
# query = f"""{column}: Is a party restricted from contracting or soliciting customers or
# partners of the counterparty, whether during the contract or after
# the contract ends (or both)?
# """
# query = f"""{column}: Is there a restriction on a party’s soliciting or hiring employees
# and/or contractors from the counterparty, whether during the contract or after the contract ends (or both)?"""
# query = f"""{column}: Is there a requirement on a party not to disparage the counterparty?"""


query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]
query_dict[column] = query

In [ ]:
query_dict

{'Governing Law': 'Governing Law: Which state/country’s law governs the interpretation of the contract?'}

### Validation

In [ ]:
pdf_file

PosixPath('/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Strategic Alliance/PLAYAHOTELS_RESORTSNV_03_14_2017-EX-10.22-STRATEGIC ALLIANCE AGREEMENT (Hyatt Ziva Cancun).PDF')

In [ ]:
pdf_files_paths[187]

PosixPath('/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Strategic Alliance/PLAYAHOTELS_RESORTSNV_03_14_2017-EX-10.22-STRATEGIC ALLIANCE AGREEMENT (Hyatt Ziva Cancun).PDF')

In [ ]:
pdf_files_paths[189]


PosixPath('/content/drive/MyDrive/RAG/data/CUAD_v1/full_contract_pdf/Part_I/Strategic Alliance/MOELIS_CO_03_24_2014-EX-10.19-STRATEGIC ALLIANCE AGREEMENT.PDF')

In [ ]:
val_chunks_no_enc= {}
mmr = {}
n_docs = 0

for pdf_file in track(pdf_files_paths[:200], description="Processing..."):
    pdf_filename = pdf_file.name
    name = Path(pdf_file).stem.replace(" ", "-")
    vector_data = collection.get(where={"source": name}, include = ['embeddings', 'documents'])

    chunks = vector_data["documents"]
    retrieved_chunks = hybrid_search(query, query_embedding, chunks, collection, name, use_cross_enc = False, top_k = 5)
    try:
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
    except:
        pdf_filename = Path(pdf_filename).stem + '.PDF'
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
    answers = ast.literal_eval(answers)
    if not answers == []:
        n_docs += 1
        mmr[pdf_filename] = 0.
        for ind, chunk in enumerate(retrieved_chunks):
            result = []
            for answer in answers:
                result.append(is_phrase_in_text(answer, chunk)['contained'])
            if all(result) and mmr[pdf_filename] == 0.:
                # print(pdf_filename, ind)
                val_chunks_no_enc[pdf_filename] = chunk
                mmr[pdf_filename] = 1./(ind + 1)
    # print('done with file:', pdf_filename)

Output()

IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
print('num docs:', n_docs)
print('Hit rate@5:', np.round(len(val_chunks_no_enc.keys())/n_docs, 2))
print('Mean Reciprocal Rank:', np.mean(list(mmr.values())))

num docs: 119
Hit rate@5: 0.93
Mean Reciprocal Rank: 0.8648459383753502


In [ ]:
hit_rate_dict[column] = len(val_chunks_no_enc.keys())/n_docs
mrr_dict[column] = np.mean(list(mmr.values()))

In [ ]:
query_df = pd.DataFrame(query_dict.items(), columns=['Column', 'Query'])
hit_rate_df = pd.DataFrame(hit_rate_dict.items(), columns=['Column', 'Hit Rate'])
mrr_df = pd.DataFrame(mrr_dict.items(), columns=['Column', 'Mean Reciprocal Rank'])

query_df.to_csv('query.csv', index=False)
hit_rate_df.to_csv('hit_rate.csv', index=False)
mrr_df.to_csv('mrr.csv', index=False)

In [ ]:
print('average hit rate:', np.mean(list(hit_rate_dict.values())))
print('average mrr:', np.mean(list(mrr_dict.values())))

average hit rate: 0.602686047340913
average mrr: 0.4918697267235403


In [ ]:
hit_rate_df.tail()

,Column,Hit Rate
9,Notice Period To Terminate Renewal,1.000000
10,Most Favored Nation,0.200000
11,Competitive Restriction Exception,0.470588
12,Non-Compete,0.225806
13,Exclusivity,0.241379


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
mrr_df.tail()

,Column,Mean Reciprocal Rank
9,Notice Period To Terminate Renewal,0.680000
10,Most Favored Nation,0.200000
11,Competitive Restriction Exception,0.387255
12,Non-Compete,0.185484
13,Exclusivity,0.201149


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


# Answer Generation

In [ ]:
openaikey = '**'

In [ ]:
os.environ['OPENAI_API_KEY'] = openaikey

In [ ]:
# messages = [
# {"role": "system", "content": """Answer the question based only on the provided context.
# If the provided chunks do not contain the answer, only return "Information not available".
# """},
# {"role": "user", "content": f"Context: {retrieved_chunks[0]}\n\nQuestion: {query}"}
# ]

In [ ]:
def build_prompt(query: str, chunks: list) -> list[dict]:
    # Handle both plain strings and RetrievedChunk objects

    if not chunks:
        return [
            {"role": "user", "content": f"No relevant context was found."}
        ]

    context = chunks[0]
    messages = [
    {"role": "system", "content": """You are an assistant that extracts specific facts from legal text.
    Answer the question based only on the provided context.
    If the information is not explicitly or implicitly mentioned, return 'Information not available'.
    Be concise.
        """},
    {"role": "user", "content": f"Context: {context}\n\nQuestion: {query}"}
    ]
    return messages

In [ ]:
query = query_dict[column]
query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

print(column)
print(query)

Governing Law
Governing Law: Which state/country’s law governs the interpretation of the contract?


In [ ]:
chat_model = "Qwen/Qwen2.5-7B-Instruct"

In [ ]:
chat_client = InferenceClient(
    model=chat_model,
    token=hf_token
)

In [ ]:
llm_responses = {}
retrieved_chunk_dict = {}
correct_chunk_dict = {}
answers_dict = {}

for pdf_file in pdf_files_paths[60:61]:
    pdf_filename = pdf_file.name
    name = Path(pdf_file).stem.replace(" ", "-")
    vector_data = collection.get(where={"source": name}, include = ['embeddings', 'documents'])

    chunks = vector_data["documents"]
    retrieved_chunks = hybrid_search(query, query_embedding, chunks, collection, name, use_cross_enc = False, top_k = 1)
    retrieved_chunk_dict[pdf_filename] = retrieved_chunks[0]
    try:
        correct_chunk = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column+'-Answer'].values[0]
    except:
        pdf_filename = Path(pdf_filename).stem + '.PDF'
        correct_chunk = master_clauses[master_clauses['Filename'] == pdf_filename][column].values[0]
        answers = master_clauses[master_clauses['Filename'] == pdf_filename][column+'-Answer'].values[0]
    correct_chunk = ast.literal_eval(correct_chunk)
    correct_chunk_dict[pdf_filename] = correct_chunk

    # answers = ast.literal_eval(answers)
    answers_dict[pdf_filename] = answers

    # print(retrieved_chunks)
    # messages = [
    # {"role": "system", "content": """Answer the question based only on the provided context.
    # If the provided chunks do not contain the answer, only return "Information not available".
    # """},
    # {"role": "user", "content": f"Context: {retrieved_chunks[0]}\n\nQuestion: {query}"}
    # ]
    messages = build_prompt(
        query          = query,
        chunks         = retrieved_chunks,
    )
    # print(retrieved_chunks[0])
    # outputs = pipe(
    # messages
    # )
    response = chat_client.chat_completion(
        messages=messages,
        max_tokens=512,
        temperature=0.7
    )
    # response = outputs[0]["generated_text"][-1]["content"]
    print(response.choices[0].message.content)
    llm_responses[pdf_filename] = response

    print(answers)
    # print(retrieved_chunks[0])
    # print(response)
    print('done with file:', pdf_filename)
    print('\n')

The validity, interpretation, and performance of the Agreement shall be controlled by and construed under the laws of the State of New York, United States of America.
New York
done with file: ScansourceInc_20190822_10-K_EX-10.38_11793958_EX-10.38_Distributor Agreement1.pdf




In [ ]:
llm_responses

{}

In [ ]:
llm_responses_df = pd.read_csv('LLM_responses.csv')
llm_responses_df.drop(columns=['Unnamed: 0.1', 'Unnamed: 0'],inplace=True)
llm_responses_df.head()

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Filename,LLM Response,Answer
0,DigitalCinemaDestinationsCorp_20111220_S-1_EX-...,Delaware,Delaware
1,LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_...,Maryland,Maryland
2,SouthernStarEnergyInc_20051202_SB-2A_EX-9_8018...,Federal Republic of Germany,Germany
3,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,Nevada,Nevada
4,CreditcardscomInc_20070810_S-1_EX-10.33_362297...,Delaware's law governs the interpretation of t...,Delaware


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# llm_responses_df['Answer'] = llm_responses_df['Filename'].map(answers_dict)

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
llm_responses_df

,Filename,LLM Response,Answer
0,DigitalCinemaDestinationsCorp_20111220_S-1_EX-...,Delaware,Delaware
1,LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_...,Maryland,Maryland
2,SouthernStarEnergyInc_20051202_SB-2A_EX-9_8018...,Federal Republic of Germany,Germany
3,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,Nevada,Nevada
4,CreditcardscomInc_20070810_S-1_EX-10.33_362297...,Delaware's law governs the interpretation of t...,Delaware
5,SteelVaultCorp_20081224_10-K_EX-10.16_3074935_...,Information not available.,Virginia
6,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-1...,New York,New York
7,UnionDentalHoldingsInc_20050204_8-KA_EX-10_334...,Information not available.,NaN
8,UsioInc_20040428_SB-2_EX-10.11_1723988_EX-10.1...,Information not available,"Virginia, Texas"
9,DeltathreeInc_19991102_S-1A_EX-10.19_6227850_E...,New York,New York


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
# llm_responses_df = pd.DataFrame(llm_responses.items(), columns=['Filename', 'LLM Response'])
# llm_responses_df.head()

In [ ]:
# llm_responses_df.to_csv('LLM_responses.csv')

In [ ]:
ind = 1
response = llm_responses_df['LLM Response'].values[ind]
retrieved_chunks = [retrieved_chunk_dict[llm_responses_df['Filename'].values[ind]]]
answer = llm_responses_df['Answer'].values[ind]
print(response)
print(answer)
print(retrieved_chunks[0])

Maryland
Maryland
7.4      This Agreement shall be construed and governed in accordance
                  with the laws of the State of Maryland  regardless of the place
                  or places of its physical execution and performance.
          


In [ ]:
ragas_scores = {}

### RAGAS

In [ ]:
for pdf_file in pdf_files_paths[:4]:
    pdf_filename = pdf_file.name

    row = llm_responses_df[llm_responses_df['Filename'] == pdf_filename]
    response = row['LLM Response'].values[0]
    retrieved_chunks = [retrieved_chunk_dict[pdf_filename]]
    answer = row['Answer'].values[0]
    if pd.isna(answer):
        answer = 'Information not available'

    data = {
        "question": [query],
        "answer": [response],
        "contexts": [retrieved_chunks],
        "ground_truth": [answer]
    }

    dataset = Dataset.from_dict(data)
    result = evaluate(
        dataset=dataset,
        metrics=[
            Faithfulness(),
            AnswerCorrectness()
        ],
        embeddings = free_embeddings,
        raise_exceptions=True
    )

    ragas_scores[pdf_filename] = result
    print('done with file:', pdf_filename)

/usr/local/lib/python3.12/dist-packages/ragas/_analytics.py:278: DeprecationWarning: evaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  result = func(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/ragas/evaluation.py:457: DeprecationWarning: aevaluate() is deprecated and will be removed in a future version. Use the @experiment decorator instead. See https://docs.ragas.io/en/latest/concepts/experiment/ for more information.
  return await aevaluate(
/usr/local/lib/python3.12/dist-packages/ragas/evaluation.py:161: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  embeddings = Lang

Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

done with file: DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate Agreement.pdf


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

done with file: LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_Affiliate Agreement.pdf


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

done with file: SouthernStarEnergyInc_20051202_SB-2A_EX-9_801890_EX-9_Affiliate Agreement.pdf


Evaluating:   0%|          | 0/2 [00:00<?, ?it/s]

done with file: CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf


In [ ]:
faithfulness_scores = {}
answer_correctness_scores = {}
for pdf_file in pdf_files_paths[:20]:
    pdf_filename = pdf_file.name

    faithfulness_scores[pdf_filename] = ragas_scores[pdf_filename]['faithfulness'][0]
    answer_correctness_scores[pdf_filename] = ragas_scores[pdf_filename]['answer_correctness'][0]

    # print('done with file:', pdf_filename)

In [ ]:
llm_responses_df['Faithfulness'] = llm_responses_df['Filename'].map(faithfulness_scores)
llm_responses_df['Answer Correctness'] = llm_responses_df['Filename'].map(answer_correctness_scores)
llm_responses_df

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


,Filename,LLM Response,Answer,Faithfulness,Answer Correctness
0,DigitalCinemaDestinationsCorp_20111220_S-1_EX-...,Delaware,Delaware,1.0,1.000000
1,LinkPlusCorp_20050802_8-K_EX-10_3240252_EX-10_...,Maryland,Maryland,1.0,1.000000
2,SouthernStarEnergyInc_20051202_SB-2A_EX-9_8018...,Federal Republic of Germany,Germany,1.0,0.946743
3,CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605...,Nevada,Nevada,1.0,1.000000
4,CreditcardscomInc_20070810_S-1_EX-10.33_362297...,Delaware's law governs the interpretation of t...,Delaware,1.0,0.917865
5,SteelVaultCorp_20081224_10-K_EX-10.16_3074935_...,Information not available.,Virginia,0.0,0.119951
6,TubeMediaCorp_20060310_8-K_EX-10.1_513921_EX-1...,New York,New York,1.0,1.000000
7,UnionDentalHoldingsInc_20050204_8-KA_EX-10_334...,Information not available.,NaN,0.0,0.987943
8,UsioInc_20040428_SB-2_EX-10.11_1723988_EX-10.1...,Information not available,"Virginia, Texas",0.0,0.088105
9,DeltathreeInc_19991102_S-1A_EX-10.19_6227850_E...,New York,New York,1.0,1.000000


In [ ]:
print('Mean Faithfulness', np.mean(list(faithfulness_scores.values())))
print('Mean Answer Correctness', np.mean(list(answer_correctness_scores.values())))

Mean Faithfulness 0.8
Mean Answer Correctness 0.8629601995427265


In [ ]:
llm_responses_df.to_csv(f'LLM_responses_{column}.csv')

# REST API

In [ ]:
import logging
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel

In [ ]:
# ── Build FAISS index from chunks ─────────────────────────────────────────────
def build_faiss_index(chunks: list[str]) -> tuple:
    embeddings = embed_chunks(chunks, embedding_model)
    embeddings_np = np.array(embeddings)

    dimension = embeddings_np.shape[1]                           # e.g. 1024 for bge-large
    index     = faiss.IndexFlatIP(dimension)                     # Inner Product = cosine sim (for normalized embeddings)
    index.add(embeddings_np)

    return index, embeddings_np


def hybrid_search_faiss(query, query_embedding, chunks, index, use_cross_enc=True, top_k=5):

    # 1. Dense retrieval — FAISS instead of ChromaDB
    query_np      = np.array([query_embedding]).astype("float32")
    distances, indices = index.search(query_np, k=10)            # top 10 nearest neighbors
    dense_ids     = [f"chunk_{i}" for i in indices[0]]     # match your existing ID format

    # 2. Sparse retrieval (BM25) — unchanged
    tokenized_chunks = [tokenizer.tokenize(chunk) for chunk in chunks]
    bm25             = BM25Okapi(tokenized_chunks)
    tokenized_query  = tokenizer.tokenize(query)
    bm25_scores      = bm25.get_scores(tokenized_query)
    top_sparse_indices = sorted(
        range(len(bm25_scores)),
        key=lambda i: bm25_scores[i],
        reverse=True)[:20]
    sparse_ids = [f"chunk_{i}" for i in top_sparse_indices]

    # 3. RRF merge — unchanged
    def reciprocal_rank_fusion(dense_ids, sparse_ids, k=60):
        scores = {}
        for rank, doc_id in enumerate(dense_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        for rank, doc_id in enumerate(sparse_ids):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank + 1)
        return sorted(scores, key=scores.get, reverse=True)

    # Helper — convert chunk ID back to text
    def ids_to_texts(ids: list[str]) -> list[str]:
        return [
            chunks[int(id.replace(f"chunk_", ""))]
            for id in ids
            if int(id.replace(f"chunk_", "")) < len(chunks)
        ]

    if use_cross_enc:
        candidate_ids   = reciprocal_rank_fusion(dense_ids, sparse_ids)[:10]
        candidate_texts = ids_to_texts(candidate_ids)

        # 4. Cross-encoder reranking — unchanged
        rerank_pairs    = [[query, text] for text in candidate_texts]
        scores          = cross_encoder_model.predict(rerank_pairs)

        doc_score_pairs = sorted(zip(candidate_texts, scores), key=lambda x: x[1], reverse=True)
        return [text for text, score in doc_score_pairs[:top_k]]

    else:
        final_ids   = reciprocal_rank_fusion(dense_ids, sparse_ids)[:top_k]
        final_texts = ids_to_texts(final_ids)
        return final_texts

In [ ]:
file_path = f'{cwd}/CybergyHoldingsInc_20140520_10-Q_EX-10.27_8605784_EX-10.27_Affiliate Agreement.pdf'

In [ ]:
def rag(pdf, query):
    text = extract_text_from_pdf(pdf)
    paragraphs = text_splitter.split(text, do_paragraph_segmentation=True)
    chunks = extract_chunks(paragraphs)
    index, _        = build_faiss_index(chunks)
    print('done with vectorizing')

    query_embedding = embedding_model.encode([query], convert_to_numpy=True)[0]

    retrieved_chunks = hybrid_search_faiss(
    query          = query,
    query_embedding= query_embedding,
    chunks         = chunks,
    index          = index,
    use_cross_enc  = True,
    top_k          = 1
    )
    print('done with retrieval')

    messages = build_prompt(
        query          = query,
        chunks         = retrieved_chunks,
    )

    outputs = pipe(
    messages
    )

    print('done with generation')

    response = outputs[0]["generated_text"][-1]["content"]
    return response

In [ ]:
query

'Governing Law: Which state/country’s law governs the interpretation of the contract?'

In [ ]:
response = rag(file_path, query)
response

done with vectorizing


Both `max_new_tokens` (=50) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


done with retrieval
done with generation


'Nevada'

In [ ]:
app    = FastAPI()
logger = logging.getLogger(__name__)

In [ ]:
class QueryRequest(BaseModel):
    text:  str
    query: str

@app.post("/query")
def query_rag(request: QueryRequest):
    try:
        answer = rag(request.text, request.query)
        return {"answer": answer}
    except Exception as e:
        logger.error(f"Query failed: {str(e)}")
        raise

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [ ]:
@app.get("/health")
def health():
    return {"status": "ok"}

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8000)

RuntimeError: asyncio.run() cannot be called from a running event loop

# Appendix

## Naive Retrieval

In [ ]:
def retrieve(query, model, index, chunks, embeddings, top_k=5):
    query_vec = model.encode([query], convert_to_numpy=True)
    D, I = index.search(query_vec, top_k)
    return [chunks[i] for i in I[0]], I[0]

In [ ]:
relevant_chunks, chunk_indices = retrieve(query, model, index, chunks, embeddings)

In [ ]:
for i, chunk in enumerate(relevant_chunks):
    print(f"Chunk {i+1}: {chunk}\n")

Chunk 1:                                                                    EXHIBIT 10.15
                              CO-BRANDING AGREEMENT
 This Agreement is made this 21st day of January 2003  by and between Lucent
Technologies Inc. , a Delaware corporation having a principal place of business
at 600 Mountain Avenue, Murray Hill , New Jersey 07974 ("Lucent") and mPhase
Technologies Inc., a New Jersey corporation located at 587 Connecticut Avenue,
Norwalk, Connecticut 068545 ("mPhase") (each individually, "a Party" and,
collectively, "the Parties"}.
 WHEREAS, mPhase wishes to use the Lucent Technologies name and Logo and the
slogan TECHNOLOGY BY LUCENT TECHNOLOGIES on printed circuit boards, product
packaging and in printed marketing materials ("Approved Uses") in connection
with its multi-access product (the "Goods") and Lucent wishes to permit mPhase
to do so.
 

Chunk 2: IN WITNESS WHEREOF, the Parties by their duly authorized representatives, have
executed this Agreement on the 

In [ ]:
client = chromadb.PersistentClient(path="./cuad_vector_db")

In [ ]:
for pdf_file in pdf_files_paths[2:60]:
    pdf_filename = pdf_file.name
    name = Path(pdf_file).stem.replace(" ", "-")
    collection = client.get_collection(name=name)
    vector_data = collection.get(include=['documents', 'embeddings'])

    ids = [name+id for id in vector_data['ids']]
    metadatas = [{'source': name}]*len(vector_data['documents'])

    new_collection.add(ids = ids,
                       documents = vector_data['documents'],
                       embeddings = vector_data['embeddings'],
                       metadatas = metadatas)
    assert vector_data['embeddings'].shape == new_collection.get(where = {'source': name}, include= ['embeddings'])['embeddings'].shape

    print(f'Finished with {name}')

In [ ]:
for file_path in pdf_files_paths[50:60]:
    # Create a collection (similar to a table in SQL)
    name = Path(file_path).stem.replace(" ", "-") # Replace spaces with hyphens for valid collection name
    collection = client.get_or_create_collection(name=name)

    text = extract_text_from_pdf(file_path)
    print('extracted text')
    paragraphs = sat.split(text, do_paragraph_segmentation=True)
    print('extracted paragraph')

    chunks = extract_chunks(paragraphs)
    embeddings = embed_chunks(chunks, embedding_model)
    print('extracted embeddings')

    collection.add(
      documents = chunks,
      embeddings = embeddings,
      ids=[f"chunk_{i}" for i in range(len(chunks))]
    )

    print(f'Finished with {name}')

### Editing old ids

In [ ]:
pdf_file = pdf_files_paths[0]
pdf_filename = pdf_file.name
name = Path(pdf_file).stem.replace(" ", "-")
name

'DigitalCinemaDestinationsCorp_20111220_S-1_EX-10.10_7346719_EX-10.10_Affiliate-Agreement'

In [ ]:
collection = client.get_collection(name=name)
existing_item = collection.get(include=["embeddings", "metadatas", "documents"])

In [ ]:
ids = [name+id for id in existing_item['ids']]
metadatas = [{'source': name}]*len(existing_item['documents'])

In [ ]:
new_collection.add(ids = ids,
                    documents = existing_item['documents'],
                    embeddings = existing_item['embeddings'],
                    metadatas = metadatas)

In [ ]:
new_collection.delete(ids=existing_item['ids'])


{'deleted': 221}